# Puffin Transcription Initiation

![Puffin Transcription Initiation](https://proto-bio.github.io/proto-assets/images/tool/puffin/hero.png)

Puffin predicts per-base transcription initiation signals from DNA sequence and decomposes them into learned motif contributions ([Dudnyk et al., *Science* 2024](https://doi.org/10.1126/science.adj0116)). Three tools share the same model: `puffin-prediction` is a fast forward pass returning all 10 channels, `puffin-interpretation` runs gradient-based motif decomposition for one chosen target signal and strand, and `puffin-gradient` reports a track's mean signal together with its gradient for sequence design.

In [ ]:
from proto_tools.utils.notebook_docs import display_api_reference, display_available_tools, display_doc_link, display_docs_section, display_overview

display_doc_link("puffin")
display_overview("puffin")
display_docs_section("puffin", "Background")

## Available tools

In [ ]:
display_available_tools("puffin")

### `run_puffin_prediction`

Fast forward pass through Puffin. For each input DNA sequence (>= 651 bp) returns per-base predictions across 10 channels (five transcription assays on both strands). Per-base output length equals `len(sequence) - 650`.

In [ ]:
from proto_tools import (
    PuffinPredictionConfig,
    PuffinPredictionInput,
    TRACK_NAMES,
    run_puffin_prediction,
)

In [ ]:
# Display input docs
display_api_reference("puffin-prediction", "input", "run_puffin_prediction")

In [ ]:
# Display config docs
display_api_reference("puffin-prediction", "config", "run_puffin_prediction")

In [ ]:
from pathlib import Path

# Real HBB promoter window (1650 bp -> 1000 bp per-base output); the HBB
# transcription start site sits at the centre of the scored region.
hbb = "".join(l for l in Path("hbb_genomic.fasta").read_text().splitlines() if not l.startswith(">"))
sequence = hbb[4176:5826]
prediction_result = run_puffin_prediction(
    PuffinPredictionInput(sequences=[sequence]),
    PuffinPredictionConfig(),
)

entry = prediction_result.results[0]
fantom_idx = TRACK_NAMES.index("FANTOM_CAGE+")
peak = max(range(entry.output_length), key=lambda i: entry.predictions[i][fantom_idx])
print(f"shape: {entry.output_length} bp x {len(entry.predictions[0])} channels")
print(f"FANTOM_CAGE+ peak at input position {entry.output_start + peak}")

In [ ]:
display_api_reference("puffin-prediction", "output", "run_puffin_prediction")

### `run_puffin_interpretation`

Gradient-based motif decomposition for one chosen target signal and strand. Returns the per-base prediction plus 18 motif activation tracks (9 motifs x 2 strands) and basepair-level contribution scores.

In [ ]:
from proto_tools import (
    MOTIF_NAMES,
    PuffinInterpretationConfig,
    PuffinInterpretationInput,
    run_puffin_interpretation,
)

In [ ]:
# Display input docs
display_api_reference("puffin-interpretation", "input", "run_puffin_interpretation")

In [ ]:
# Display config docs
display_api_reference("puffin-interpretation", "config", "run_puffin_interpretation")

In [ ]:
# Decompose the PRO_CAP forward-strand prediction
sequence = "ATCG" * 412 + "AT"
interpretation_result = run_puffin_interpretation(
    PuffinInterpretationInput(sequences=[sequence]),
    PuffinInterpretationConfig(target_signal="PRO_CAP"),
)

r = interpretation_result.results[0]
print(f"target_signal: {interpretation_result.target_signal}, reverse_strand: {interpretation_result.reverse_strand}")
print(f"motif tracks: {len(r.motif_activations)} (9 motifs x 2 strands)")
print(f"motifs: {MOTIF_NAMES}")

In [ ]:
display_api_reference("puffin-interpretation", "output", "run_puffin_interpretation")

### `run_puffin_gradient`

The gradient tool turns Puffin's initiation prediction into a differentiable quantity. Each term names an output track (with its strand suffix) and a span of output positions, and comes back with that track's mean signal and its own raw gradient; a direction and a weight are goals the caller applies afterwards.

Unlike the fixed-context genomic models, Puffin scores **every base** of a variable-length input — so a candidate is just an `(L, 4)` relaxed matrix and the gradient covers all `L` positions. Inputs need at least 651 bp, since Puffin trims 325 bp from each end and the scored window is `L - 650` positions.

Here the objective targets the FANTOM_CAGE+ peak located above, asking which base changes would raise initiation at exactly that position.

In [ ]:
from proto_tools import (
    PuffinGradientConfig, PuffinGradientInput, PuffinSignalTerm, run_puffin_gradient,
)

display_api_reference("puffin-gradient", "input", "run_puffin_gradient")

# Relax the same HBB promoter window: a sharpened one-hot of the real sequence, so the
# starting distribution favours the wild-type base at each position without saturating.
grad_sharpness = 2.0
grad_logits = [
    [grad_sharpness if base == observed else 0.0 for base in "ACGT"]
    for observed in sequence.upper()
]
print(f"relaxed input: {len(grad_logits)} bp -> {len(grad_logits) - 650} scored positions")


In [ ]:
display_api_reference("puffin-gradient", "config", "run_puffin_gradient")

# Target a 21 bp window centred on the FANTOM_CAGE+ peak found above. Term positions are
# 1-indexed and relative to the scored output window, which is what `peak` already indexes.
grad_config = PuffinGradientConfig(
    gradient_terms=[
        PuffinSignalTerm(
            track="FANTOM_CAGE+",
            position_start=max(1, peak + 1 - 10),
            position_end=min(entry.output_length, peak + 1 + 10),
        )
    ],
)

grad_output = run_puffin_gradient(PuffinGradientInput(inputs=grad_logits), grad_config)


In [ ]:
display_api_reference("puffin-gradient", "output", "run_puffin_gradient")

grad_result = grad_output.results[0]
grad_term = grad_result.terms[0]
print(f"Scored positions:   {grad_result.metrics['output_length']}")
print(f"Gradient shape:     ({len(grad_term.gradient)}, {len(grad_term.gradient[0])})")
print(f"  {grad_term.term.track} positions "
      f"{grad_term.term.position_start}-{grad_term.term.position_end}: "
      f"mean signal {grad_term.term_value:.4f}")

# The gradient covers the whole input, including the 325 bp flanks Puffin trims from the
# output -- those bases still shape the prediction, so they still carry a derivative.
grad_peak_row = grad_term.gradient[325 + peak]
print(f"Gradient at the peak base: {[round(v, 6) for v in grad_peak_row]}")
display_api_reference("puffin-gradient", "input", "run_puffin_gradient")
display_api_reference("puffin-gradient", "config", "run_puffin_gradient")
display_api_reference("puffin-gradient", "output", "run_puffin_gradient")


## Export Results

In [ ]:
from pathlib import Path

out_dir = Path("puffin_output")
out_dir.mkdir(exist_ok=True)
prediction_result.export(name="prediction", export_path=out_dir, file_format="json")
interpretation_result.export(name="interpretation", export_path=out_dir, file_format="json")